In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD
import joblib

In [ ]:
CLEAN_TMDB_FILE_PATH = "../datasets/clean/tmdb-movies/TMDB_movie_dataset_v11.csv"
CLEAN_MOVIELENS_RATINGS_PATH = "../datasets/clean/ml-32m/ratings.csv"

ML_API_TF_IDF_MATRIX_PATH = "../../ml-api/model/tf_idf_matrix.pkl"

In [ ]:
tmdb = pd.read_csv(CLEAN_TMDB_FILE_PATH)
ratings = pd.read_csv(CLEAN_MOVIELENS_RATINGS_PATH)

In [ ]:
tmdb.head()

In [ ]:
tmdb.info()

In [ ]:
ratings.head()

In [ ]:
ratings.info()

In [ ]:
tmdb_id_to_index = pd.Series(tmdb.index, index=tmdb["id"]).to_dict()
ratings["rating"] = ratings["rating"] - ratings["rating"].mean()

In [ ]:
plt.hist(ratings["rating"], bins=50)
plt.title("Distribution ratings")
plt.xlabel("Score")
plt.ylabel("Frequency")
plt.show()

In [ ]:
print("min:", ratings["rating"].min())
print("max:", ratings["rating"].max())
print("mean:", ratings["rating"].mean())
print("std:", ratings["rating"].std())
print(np.percentile(ratings["rating"], [0, 25, 50, 75, 100]))

In [ ]:
def find_in_dataset_by_substring(movie_name):
    results = tmdb[tmdb["title"].str.contains(movie_name, case=False, na=False)]
    return results.sort_values(by="popularity", ascending=False)

def find_in_dataset_by_id(movie_id):
    return tmdb[tmdb["id"] == movie_id]["title"]

In [ ]:
find_in_dataset_by_substring("twilight")

In [ ]:
find_in_dataset_by_id(453395)

In [ ]:
spiderman_fan_collaborative = {
  569094: 4.5 - 3.5,
  634649: 4 - 3.5,
  324857: 5 - 3.5,
  524434: 1 - 3.5,
  18224: 0.5 - 3.5
}

for id in spiderman_fan_collaborative.keys():
  print(f"{id}: {find_in_dataset_by_id(id)}")

Content based filtering

In [ ]:
tfidf = TfidfVectorizer(
    max_features=50000,
    stop_words="english"
)

tfidf_matrix = tfidf.fit_transform(tmdb["content"])

In [ ]:
def build_user_profile_content(ratings_dict):
    vectors = []
    weights = []

    for tmdb_id, rating in ratings_dict.items():
        if tmdb_id in tmdb_id_to_index:
            idx = tmdb_id_to_index[tmdb_id]
            vectors.append(tfidf_matrix[idx].toarray()[0])
            weights.append(rating + 3.5)

    vectors = np.array(vectors)
    weights = np.array(weights)

    # vážený průměr
    user_profile = np.average(vectors, axis=0, weights=weights)
    user_profile = user_profile.reshape(1, -1)

    return user_profile

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend_content(user_vector, top_k=20):
    sims = cosine_similarity(user_vector, tfidf_matrix).flatten()
    top_idx = sims.argsort()[-top_k:][::-1]
    return tmdb.iloc[top_idx][["title", "vote_average"]]

In [ ]:
spiderman_fan = build_user_profile_content(spiderman_fan_collaborative)

recommend_content(spiderman_fan)

Collaborative filtering

In [ ]:
ratings[ratings["tmdbId"].isna()]

In [ ]:
ratings[ratings["userId"].isna()]

In [ ]:
movie_stats = ratings.groupby("tmdbId").agg(
    avg=("rating", "mean"),
    count=("rating", "count")
)

good_movies = movie_stats[
    (movie_stats["count"] >= 0)
].index

ratings_filtered = ratings[ratings["tmdbId"].isin(good_movies)]

In [ ]:
len(np.sort(ratings["tmdbId"].unique()))

In [ ]:
ratings_test = ratings_filtered


user_ids = np.sort(ratings_test["userId"].unique())
movie_ids = np.sort(ratings_test["tmdbId"].unique())

user_map = {u: i for i, u in enumerate(user_ids)}
movie_map = {m: i for i, m in enumerate(movie_ids)}

n_users = len(user_ids)
n_items = len(movie_ids)

rows = ratings_test["userId"].map(user_map)
cols = ratings_test["tmdbId"].map(movie_map)
data = ratings_test["rating"]

R = csr_matrix((data, (rows, cols)), shape=(n_users, n_items))

svd = TruncatedSVD(n_components=35)
U = svd.fit_transform(R)
V = svd.components_

In [ ]:
U

In [ ]:
V

In [ ]:
_indices = []
_ratings = []

for tmdb_id, rating in spiderman_fan_collaborative.items():
    if tmdb_id in movie_map:
        _indices.append(movie_map[tmdb_id])
        _ratings.append(rating)
    
V_sub = V[:, _indices]
r = np.array(_ratings)
user_vector = np.linalg.lstsq(V_sub.T, r, rcond=None)[0]
scores = user_vector @ V
scores

In [ ]:
print("min:", scores.min())
print("max:", scores.max())
print("mean:", scores.mean())
print("std:", scores.std())
print(np.percentile(scores, [0, 25, 50, 75, 100]))

In [ ]:
plt.hist(scores, bins=50)
plt.title("Distribution of predicted scores")
plt.xlabel("Score")
plt.ylabel("Frequency")
plt.show()

In [ ]:
scores_cut = np.clip(scores, -5, 10)
plt.hist(scores_cut, bins=50)
plt.title("Distribution of predicted scores")
plt.xlabel("Score")
plt.ylabel("Frequency")
plt.show()

In [ ]:
top_k = 100

top_idx = scores.argsort()[::-1][:top_k]
top_movies = [(movie_ids[i], scores[i]) for i in top_idx]

results = []

tmdb_indexed = tmdb.set_index("id")

for movie_id, score in top_movies:
    if movie_id in tmdb_indexed.index:
        title = tmdb_indexed.loc[movie_id]["title"]
        results.append((title, score))

results

Saving this abhorrent "progress"

In [ ]:
joblib.dump(tfidf_matrix, ML_API_TF_IDF_MATRIX_PATH)